# 04 – Modelling, Tuning and Evaluation
Predicting used car prices (log price as target) with a baseline, Ridge regression, Random Forest and HistGradientBoosting.

### Data and train/test split
I used year, odometer, cylinders and 11 categorical features. I dropped `car_age` because it duplicates `year` (correlation -1.00), and `has_vin` and `description_length` because an app user can't enter them. The target is log price, since raw price has skewness 2.0. I kept 20% of the data as a test set and used it only once, at the end. The `model` column has thousands of values, so I kept the 100 most common (chosen on training data only) and grouped the rest as "other".

In [1]:
import pandas as pd, numpy as np, joblib, os, json, gc
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

num_cols = ["year", "odometer", "cylinders"]
cat_cols = ["manufacturer", "model", "condition", "fuel", "transmission", "drive",
            "type", "paint_color", "title_status", "seller_type", "state"]

# Load only the needed columns (skips description, VIN, URLs and so on)
df = pd.read_parquet("../data/processed/vehicles_features.parquet",
                     columns=num_cols + cat_cols + ["log_price"])
X = df[num_cols + cat_cols].copy()
X["cylinders"] = X["cylinders"].astype(float)
X[cat_cols] = X[cat_cols].astype(str)
y = df["log_price"]
del df; gc.collect()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test = X_train.copy(), X_test.copy()
del X; gc.collect()

top_models = X_train["model"].value_counts().head(100).index
for d in (X_train, X_test):
    d["model"] = d["model"].where(d["model"].isin(top_models), "other")
print(X_train.shape, X_test.shape)

(171402, 14) (42851, 14)


### Models and preprocessing
Ridge uses scaled numbers and one-hot encoded categories. The tree models use ordinal encoding because they don't need scaling. Missing numbers are filled with the median inside the pipeline, so nothing leaks from the test data. I compared a mean-prediction baseline, Ridge (a simple linear model), Random Forest (many averaged decision trees) and HistGradientBoosting (trees built one after another, each fixing the previous errors).

In [2]:
def make_pre(kind):
    if kind == "linear":
        num = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
        cat = OneHotEncoder(handle_unknown="ignore", min_frequency=100)
    else:
        num = Pipeline([("imp", SimpleImputer(strategy="median"))])
        cat = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    return ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])

models = {
    "Baseline (mean)": Pipeline([("pre", make_pre("linear")), ("m", DummyRegressor())]),
    "Ridge": Pipeline([("pre", make_pre("linear")), ("m", Ridge(alpha=1.0))]),
    "Random Forest": Pipeline([("pre", make_pre("tree")),
        ("m", RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5,
                                    n_jobs=-1, random_state=42))]),
    "HistGradientBoosting": Pipeline([("pre", make_pre("tree")),
        ("m", HistGradientBoostingRegressor(random_state=42))]),
}

### Model comparison with cross-validation
3-fold cross-validation on a 50,000-row training sample (RMSE is on log price). The mean baseline gave RMSE 0.866 and R² of 0.00, Ridge 0.475 (R² 0.70), Random Forest 0.440 (0.74) and HistGradientBoosting 0.415 (0.77). The tree models beat Ridge clearly, which suggests the price relationships are not linear.

In [3]:
idx = X_train.sample(50000, random_state=42).index
Xs, ys = X_train.loc[idx], y_train.loc[idx]

rows = []
for name, m in models.items():
    cv = cross_validate(m, Xs, ys, cv=3, scoring=("neg_root_mean_squared_error", "r2"))
    rows.append({"model": name,
                 "RMSE (log price)": -cv["test_neg_root_mean_squared_error"].mean(),
                 "R2": cv["test_r2"].mean()})
    print("done:", name)
results = pd.DataFrame(rows).round(3)
results

done: Baseline (mean)
done: Ridge
done: Random Forest
done: HistGradientBoosting


,model,RMSE (log price),R2
0,Baseline (mean),0.866,-0.000
1,Ridge,0.475,0.700
2,Random Forest,0.440,0.741
3,HistGradientBoosting,0.415,0.771


### Hyperparameter tuning
I used RandomizedSearchCV with 3-fold cross-validation: 15 combinations for HistGradientBoosting and 8 for Random Forest, because random search covers a wide range faster than a full grid on this much data. Best HGB: learning rate 0.1, 400 iterations, no depth limit, minimum leaf size 20, no L2 regularisation. Best RF: 200 trees, max depth 20, minimum leaf size 2, 30% of features per split. Tuning improved the CV RMSE from 0.415 to 0.395 for HGB and from 0.440 to 0.424 for RF.

In [4]:
hgb_params = {"m__learning_rate": [0.03, 0.05, 0.1, 0.2], "m__max_iter": [200, 400, 600],
              "m__max_depth": [None, 6, 10, 15], "m__min_samples_leaf": [20, 50, 100],
              "m__l2_regularization": [0, 0.1, 1.0]}
rf_params = {"m__n_estimators": [100, 200], "m__max_depth": [15, 20, 30],
             "m__min_samples_leaf": [2, 5, 10], "m__max_features": [0.3, 0.5, 1.0]}

hgb_search = RandomizedSearchCV(models["HistGradientBoosting"], hgb_params, n_iter=15, cv=3,
    scoring="neg_root_mean_squared_error", random_state=42, verbose=1)
hgb_search.fit(Xs, ys)
print("HGB best:", hgb_search.best_params_, round(-hgb_search.best_score_, 3))

rf_search = RandomizedSearchCV(models["Random Forest"], rf_params, n_iter=8, cv=3,
    scoring="neg_root_mean_squared_error", random_state=42, verbose=1)
rf_search.fit(Xs, ys)
print("RF best:", rf_search.best_params_, round(-rf_search.best_score_, 3))

def to_py(o):
    return o.item() if hasattr(o, "item") else str(o)

os.makedirs("../models", exist_ok=True)
json.dump({"hgb": hgb_search.best_params_, "rf": rf_search.best_params_},
          open("../models/best_params.json", "w"), default=to_py, indent=2)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


HGB best: {'m__min_samples_leaf': 20, 'm__max_iter': 400, 'm__max_depth': None, 'm__learning_rate': 0.1, 'm__l2_regularization': 0} 0.395
Fitting 3 folds for each of 8 candidates, totalling 24 fits
RF best: {'m__n_estimators': 200, 'm__min_samples_leaf': 2, 'm__max_features': 0.3, 'm__max_depth': 20} 0.424


### Final results on the test set
| Model | MAE ($) | RMSE ($) | R² | Median % error |
|---|---|---|---|---|
| Ridge | 4,919 | 8,754 | 0.66 | 22.0 |
| HGB default | 3,895 | 6,709 | 0.80 | 18.6 |
| HGB tuned | 3,469 | 6,027 | 0.84 | 16.5 |
| RF default | 3,545 | 6,292 | 0.82 | 17.1 |
| RF tuned | 3,443 | 6,179 | 0.83 | 16.6 |

Tuning cut HGB's MAE from $3,895 to $3,469 (about 11%). Ridge is the weakest, which confirms the relationships are non-linear. RF tuned and HGB tuned are very close. Random Forest was trained on only 80,000 rows because of memory limits, so the comparison isn't perfectly fair. RMSE is much larger than MAE, which means a few expensive cars are predicted badly.

### Final model choice
I chose tuned HistGradientBoosting. It had the best RMSE and R², its MAE was only $26 behind Random Forest, and it was trained on all the data. Its file is also much smaller, which suits GitHub and Streamlit deployment. On the test set it is typically off by about 16.5%, or $3,469 on average. The model file is 0.6 MB.

In [6]:
params = json.load(open("../models/best_params.json"))   # or use the dict above

hgb_tuned = clone(models["HistGradientBoosting"]).set_params(**params["hgb"])
rf_tuned = clone(models["Random Forest"]).set_params(**params["rf"]).set_params(m__n_jobs=1)
rf_default = clone(models["Random Forest"]).set_params(m__n_jobs=1)

# Random forests are memory-heavy, so they are fitted on an 80k training subsample
sub = X_train.sample(80000, random_state=42).index

candidates = [
    ("Ridge", models["Ridge"], X_train.index),
    ("HGB default", models["HistGradientBoosting"], X_train.index),
    ("HGB tuned", hgb_tuned, X_train.index),
    ("RF default (80k rows)", rf_default, sub),
    ("RF tuned (80k rows)", rf_tuned, sub),
]

true = np.expm1(y_test)
final_rows = []
for name, est, rows in candidates:
    est = clone(est).fit(X_train.loc[rows], y_train.loc[rows])
    pred = np.expm1(est.predict(X_test))
    final_rows.append({"model": name,
        "MAE ($)": mean_absolute_error(true, pred),
        "RMSE ($)": root_mean_squared_error(true, pred),
        "R2 ($ scale)": r2_score(true, pred),
        "Median % error": (np.abs(pred - true) / true).median() * 100})
    del est, pred; gc.collect()
    print("done:", name)

final = pd.DataFrame(final_rows).round(2)
final

done: Ridge
done: HGB default
done: HGB tuned
done: RF default (80k rows)
done: RF tuned (80k rows)


,model,MAE ($),RMSE ($),R2 ($ scale),Median % error
0,Ridge,4919.48,8753.94,0.66,21.99
1,HGB default,3894.85,6708.78,0.80,18.55
2,HGB tuned,3469.05,6027.32,0.84,16.54
3,RF default (80k rows),3545.31,6292.17,0.82,17.12
4,RF tuned (80k rows),3443.34,6178.82,0.83,16.62


In [7]:
best_name = "HGB tuned"          
best_est = clone(hgb_tuned).fit(X_train, y_train)   

joblib.dump(best_est, "../models/price_model.joblib", compress=3)
joblib.dump(list(top_models), "../models/top_models.joblib")
print(round(os.path.getsize("../models/price_model.joblib") / 1e6, 1), "MB")

0.6 MB


## Summary
The tuned HistGradientBoosting model explains 84% of the variation in price, with an average error of about $3,500 against a median price of 12,995. It is useful as a guide for pricing, but not accurate enough to replace a dealer's judgement, especially for expensive or rare cars.